# LLaMA-2-7B: W-BFP4 / Top-2-Capped A-BiE4 G16 PPL

This notebook evaluates a bounded-outlier hybrid format. Decoder Linear weights use one shared exponent per Group-16 block. Activations first form signed-tensor `mean(X) + 3 * std(X)` candidates, then encode at most the two largest-magnitude candidates in each Group-16 block with the outlier exponent. Remaining candidates are demoted to the normal set before exponent selection.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import math
import os
import platform
import time
from dataclasses import asdict, dataclass
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"

FP16_BASELINE_PPL = 5.472103118896484
WA_BIE4_BASELINE_PPL = 5.9739089012146
ACTIVATION_BIE4_BASELINE_PPL = 6.013384819030762
EXPECTED_LINEAR_LAYERS = 224
EXPECTED_EVALUATED_BLOCKS = 166
EXPECTED_LOSS_TOKENS = 339_802
EXPECTED_WEIGHT_VALUES = 6_476_005_376
EXPECTED_ACTIVATION_VALUES = 387_117_481_984


@dataclass(frozen=True)
class HybridConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3
    rounding: str = "nearest"
    activation_threshold_method: str = "signed_mean_plus_sigma_k_std"
    sigma_k: float = 3.0
    max_outliers_per_block: int = 2
    topk_tie_break: str = "lowest_k_index"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This notebook is fixed to Group-16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This notebook is fixed to signed E5 shared exponents.")
        if self.mantissa_bits != 3:
            raise ValueError("BFP4/BiE4 require 1 sign bit + 3 magnitude bits.")
        if self.rounding != "nearest":
            raise ValueError("This notebook is fixed to nearest rounding.")
        if self.activation_threshold_method != "signed_mean_plus_sigma_k_std":
            raise ValueError("Unexpected activation threshold method.")
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError("sigma_k must be finite and non-negative.")
        if self.max_outliers_per_block != 2:
            raise ValueError("This notebook is fixed to at most two outliers per block.")
        if self.topk_tie_break != "lowest_k_index":
            raise ValueError("Unexpected Top-2 tie-break policy.")
        if min(self.weight_chunk_rows, self.activation_chunk_rows) <= 0:
            raise ValueError("Chunk sizes must be positive.")


CONFIG = HybridConfig()
CONFIG.validate()
OUTPUT_PATH = Path(
    "w-bfp4-a-bie4-top2cap-g16-signed-mu3sigma-no-lm-head-s2048.json"
)

PRIVATE_VALUE_BITS = 1 + CONFIG.mantissa_bits
WEIGHT_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS
    + CONFIG.shared_exponent_bits / CONFIG.block_size
)
ACTIVATION_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS
    + 1
    + 2 * CONFIG.shared_exponent_bits / CONFIG.block_size
)

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f"Hybrid config: {CONFIG}")
print(f"Weight storage: {WEIGHT_BITS_PER_VALUE:.4f} bits/value")
print(f"Activation storage: {ACTIVATION_BITS_PER_VALUE:.4f} bits/value")
print(f"Output: {OUTPUT_PATH.resolve()}")
if not torch.cuda.is_available():
    print("CUDA is unavailable: synthetic checks can run, but full PPL cannot.")

## Top-2-capped quantization

- Weight: BFP4, one signed E5 shared exponent per contiguous G16 block.
- Activation: BiE4, two signed E5 shared exponents and one logical type bit per value.
- Activation threshold: one signed-tensor `mean(X) + 3 * std(X)` threshold per Linear input tensor, computed before row chunking.
- Candidate: `abs(x) > threshold`.
- Encoded outlier: at most the two largest-magnitude candidates per contiguous G16 block; ties prefer the lower K index.
- Demoted candidate: a candidate outside the Top-2 that is reassigned to normal before the normal shared exponent is computed.

In [ ]:
ACTIVATION_COUNT_NAMES = (
    "total_values",
    "candidate_values",
    "encoded_outlier_values",
    "demoted_values",
    "total_blocks",
    "candidate_affected_blocks",
    "encoded_affected_blocks",
    "cap_triggered_blocks",
    "encoded_normal_only_blocks",
    "encoded_outlier_only_blocks",
    "encoded_mixed_blocks",
)
WEIGHT_COUNT_NAMES = ("total_values", "total_blocks")


def _empty_activation_counts(device):
    return torch.zeros(
        len(ACTIVATION_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _empty_weight_counts(device):
    return torch.zeros(
        len(WEIGHT_COUNT_NAMES), dtype=torch.int64, device=device
    )


@torch.no_grad()
def _signed_tensor_threshold(tensor, sigma_k, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0:
        raise ValueError("Cannot compute a threshold for an empty tensor.")
    if chunk_rows <= 0:
        raise ValueError("chunk_rows must be positive.")

    running_count = 0
    running_mean = torch.zeros((), dtype=torch.float64, device=flat.device)
    running_m2 = torch.zeros((), dtype=torch.float64, device=flat.device)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        values = flat[start:end].float()
        chunk_count = values.numel()
        chunk_var, chunk_mean = torch.var_mean(values, unbiased=False)
        chunk_mean = chunk_mean.to(torch.float64)
        chunk_var = chunk_var.to(torch.float64)

        if running_count == 0:
            running_mean = chunk_mean
            running_m2 = chunk_var * chunk_count
            running_count = chunk_count
            continue

        combined_count = running_count + chunk_count
        delta = chunk_mean - running_mean
        running_mean = running_mean + delta * (chunk_count / combined_count)
        running_m2 = (
            running_m2
            + chunk_var * chunk_count
            + delta.square() * running_count * chunk_count / combined_count
        )
        running_count = combined_count

    variance = (running_m2 / running_count).clamp_min(0.0)
    threshold = running_mean + sigma_k * torch.sqrt(variance)
    return threshold.to(torch.float32)


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


@torch.no_grad()
def _quantize_weight_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    output = (mantissa * step).reshape(flat.size(0), padded_width)
    return output[:, :width].reshape(original_shape).to(rows.dtype)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    counts = _empty_weight_counts(weight.device)
    blocks_per_row = math.ceil(weight.size(-1) / config.block_size)

    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        rows = weight[start:end]
        rows.copy_(_quantize_weight_bfp_rows(rows, config))
        chunk_counts = torch.tensor(
            [rows.numel(), rows.size(0) * blocks_per_row],
            dtype=torch.int64,
            device=weight.device,
        )
        counts.add_(chunk_counts)

    return counts


def _select_top2_candidates(magnitude, candidate, config):
    if magnitude.shape != candidate.shape:
        raise ValueError("Magnitude and candidate masks must have the same shape.")
    if magnitude.size(-1) != config.block_size:
        raise ValueError("Top-2 selection expects one complete block per last dimension.")

    selected = torch.zeros_like(candidate)
    remaining = candidate.clone()
    positions = torch.arange(config.block_size, device=magnitude.device)
    positions = positions.view(
        *([1] * (magnitude.ndim - 1)), config.block_size
    )

    for _ in range(config.max_outliers_per_block):
        has_candidate = remaining.any(dim=-1, keepdim=True)
        score = magnitude.masked_fill(~remaining, float("-inf"))
        index = score.argmax(dim=-1, keepdim=True)
        picked = (positions == index) & has_candidate
        selected = selected | picked
        remaining = remaining & ~picked

    return selected


@torch.no_grad()
def _quantize_activation_bie_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    candidate = valid_blocks & (magnitude > threshold)
    outlier = _select_top2_candidates(magnitude, candidate, config)
    demoted = candidate & ~outlier
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)

    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)
    normal_exp = torch.where(
        ~normal_present & outlier_present, outlier_exp, normal_exp
    )
    outlier_exp = torch.where(
        ~outlier_present & normal_present, normal_exp, outlier_exp
    )
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)

    real_block = valid_blocks.any(dim=-1)
    valid_count = valid_blocks.sum(dim=-1)
    candidate_count = candidate.sum(dim=-1)
    outlier_count = outlier.sum(dim=-1)
    candidate_affected = real_block & (candidate_count > 0)
    encoded_affected = real_block & (outlier_count > 0)
    cap_triggered = real_block & (
        candidate_count > config.max_outliers_per_block
    )
    outlier_only = real_block & (outlier_count == valid_count)
    mixed = encoded_affected & ~outlier_only
    candidate_histogram = torch.bincount(
        candidate_count[real_block], minlength=config.block_size + 1
    )
    encoded_histogram = torch.bincount(
        outlier_count[real_block], minlength=config.block_size + 1
    )
    counts = torch.stack(
        (
            torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
            candidate.sum(dtype=torch.int64),
            outlier.sum(dtype=torch.int64),
            demoted.sum(dtype=torch.int64),
            real_block.sum(dtype=torch.int64),
            candidate_affected.sum(dtype=torch.int64),
            encoded_affected.sum(dtype=torch.int64),
            cap_triggered.sum(dtype=torch.int64),
            (real_block & ~encoded_affected).sum(dtype=torch.int64),
            outlier_only.sum(dtype=torch.int64),
            mixed.sum(dtype=torch.int64),
        )
    )
    return dequantized, counts, candidate_histogram, encoded_histogram


@torch.no_grad()
def quantize_activation_bie(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    threshold = _signed_tensor_threshold(flat, config.sigma_k, chunk_rows)
    output = torch.empty_like(flat)
    counts = _empty_activation_counts(flat.device)
    candidate_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    encoded_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        (
            quantized,
            chunk_counts,
            chunk_candidate_histogram,
            chunk_encoded_histogram,
        ) = (
            _quantize_activation_bie_rows_with_threshold(
                flat[start:end], threshold, config
            )
        )
        output[start:end] = quantized
        counts.add_(chunk_counts)
        candidate_histogram.add_(chunk_candidate_histogram)
        encoded_histogram.add_(chunk_encoded_histogram)

    return (
        output.reshape_as(tensor),
        threshold,
        counts,
        candidate_histogram,
        encoded_histogram,
    )

## Linear replacement and exact capped-occupancy accounting

Candidate histogram index `k` is the number of threshold candidates before capping. Encoded histogram index `k` is the final number of outliers after deterministic Top-2 selection. Counters stay on the model device during evaluation and are copied to CPU only when exporting the final JSON.

In [ ]:
def _rate(numerator, denominator):
    return {
        "numerator": int(numerator),
        "denominator": int(denominator),
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts, names):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


def _histogram_percentile(histogram, q, first_bin=0):
    if not 0.0 <= q <= 1.0:
        raise ValueError("q must be in [0, 1].")
    total = sum(histogram[first_bin:])
    if total == 0:
        return None
    rank = max(1, math.ceil(q * total))
    cumulative = 0
    for value in range(first_bin, len(histogram)):
        cumulative += histogram[value]
        if cumulative >= rank:
            return value
    raise RuntimeError("Histogram percentile closure failed.")


def _summarize_activation(
    counts, candidate_histogram, encoded_histogram, include_tail
):
    candidate_histogram = [int(value) for value in candidate_histogram]
    encoded_histogram = [int(value) for value in encoded_histogram]
    total_blocks = counts["total_blocks"]
    candidate_values = counts["candidate_values"]
    encoded_values = counts["encoded_outlier_values"]
    demoted_values = counts["demoted_values"]
    candidate_affected = counts["candidate_affected_blocks"]
    encoded_affected = counts["encoded_affected_blocks"]
    cap_triggered = counts["cap_triggered_blocks"]

    expected_length = CONFIG.block_size + 1
    if len(candidate_histogram) != expected_length:
        raise RuntimeError("Unexpected candidate histogram length.")
    if len(encoded_histogram) != expected_length:
        raise RuntimeError("Unexpected encoded histogram length.")
    if sum(candidate_histogram) != total_blocks:
        raise RuntimeError("Candidate histogram does not close over blocks.")
    if sum(encoded_histogram) != total_blocks:
        raise RuntimeError("Encoded histogram does not close over blocks.")
    if sum(
        index * value for index, value in enumerate(candidate_histogram)
    ) != candidate_values:
        raise RuntimeError("Candidate histogram does not close over values.")
    if sum(
        index * value for index, value in enumerate(encoded_histogram)
    ) != encoded_values:
        raise RuntimeError("Encoded histogram does not close over values.")
    if candidate_values - encoded_values != demoted_values:
        raise RuntimeError("Candidate/encoded/demoted value closure failed.")
    if candidate_histogram[0] != total_blocks - candidate_affected:
        raise RuntimeError("Candidate zero-bin closure failed.")
    if sum(candidate_histogram[1:]) != candidate_affected:
        raise RuntimeError("Candidate affected-block closure failed.")
    if encoded_histogram[0] != counts["encoded_normal_only_blocks"]:
        raise RuntimeError("Encoded zero-bin closure failed.")
    if sum(encoded_histogram[1:]) != encoded_affected:
        raise RuntimeError("Encoded affected-block closure failed.")
    if candidate_affected != encoded_affected:
        raise RuntimeError("A nonempty candidate set must encode an outlier.")
    if sum(
        candidate_histogram[CONFIG.max_outliers_per_block + 1 :]
    ) != cap_triggered:
        raise RuntimeError("Cap-triggered block closure failed.")
    if sum(encoded_histogram[CONFIG.max_outliers_per_block + 1 :]) != 0:
        raise RuntimeError("Encoded occupancy exceeds the configured cap.")
    if (
        counts["encoded_outlier_only_blocks"]
        + counts["encoded_mixed_blocks"]
        != encoded_affected
    ):
        raise RuntimeError("Encoded affected-block partition failed.")

    max_candidate = max(
        (index for index, value in enumerate(candidate_histogram) if value),
        default=None,
    )
    max_encoded = max(
        (index for index, value in enumerate(encoded_histogram) if value),
        default=None,
    )
    percentiles = (
        ("50", 0.50),
        ("90", 0.90),
        ("95", 0.95),
        ("99", 0.99),
        ("99_9", 0.999),
        ("99_99", 0.9999),
    )
    summary = {
        "counts": counts,
        "candidate_histogram_semantics": (
            "index k = blocks containing exactly k threshold candidates before capping"
        ),
        "candidate_outliers_per_block_histogram": candidate_histogram,
        "encoded_histogram_semantics": (
            "index k = blocks containing exactly k encoded outliers after Top-2 capping"
        ),
        "encoded_outliers_per_block_histogram": encoded_histogram,
        "max_candidate_outliers_per_block": max_candidate,
        "max_encoded_outliers_per_block": max_encoded,
        "mean_candidate_outliers_per_all_blocks": (
            None if total_blocks == 0 else candidate_values / total_blocks
        ),
        "mean_encoded_outliers_per_all_blocks": (
            None if total_blocks == 0 else encoded_values / total_blocks
        ),
        "mean_demoted_values_per_cap_triggered_block": (
            None if cap_triggered == 0 else demoted_values / cap_triggered
        ),
        "nearest_rank_percentiles": {
            "candidate_all_blocks": {
                f"p{label}": _histogram_percentile(candidate_histogram, q)
                for label, q in percentiles
            },
            "candidate_affected_blocks_only": {
                f"p{label}": _histogram_percentile(
                    candidate_histogram, q, first_bin=1
                )
                for label, q in percentiles
            },
            "encoded_all_blocks": {
                f"p{label}": _histogram_percentile(encoded_histogram, q)
                for label, q in percentiles
            },
        },
        "rates": {
            "candidate_value_rate": _rate(
                candidate_values, counts["total_values"]
            ),
            "encoded_outlier_value_rate": _rate(
                encoded_values, counts["total_values"]
            ),
            "demoted_value_rate": _rate(
                demoted_values, counts["total_values"]
            ),
            "demoted_fraction_of_candidates": _rate(
                demoted_values, candidate_values
            ),
            "candidate_affected_block_rate": _rate(
                candidate_affected, total_blocks
            ),
            "encoded_affected_block_rate": _rate(
                encoded_affected, total_blocks
            ),
            "cap_triggered_block_rate": _rate(cap_triggered, total_blocks),
            "encoded_normal_only_block_rate": _rate(
                counts["encoded_normal_only_blocks"], total_blocks
            ),
            "encoded_outlier_only_block_rate": _rate(
                counts["encoded_outlier_only_blocks"], total_blocks
            ),
            "encoded_mixed_block_rate": _rate(
                counts["encoded_mixed_blocks"], total_blocks
            ),
        },
    }
    if include_tail:
        summary["candidate_tail_probabilities"] = [
            {
                "minimum_candidates": minimum,
                **_rate(sum(candidate_histogram[minimum:]), total_blocks),
            }
            for minimum in range(1, CONFIG.block_size + 1)
        ]
    return summary


class HybridLinear(nn.Module):
    def __init__(self, linear, config, weight_counts):
        super().__init__()
        self.linear = linear
        self.config = config
        self.weight_counts = _counts_to_dict(
            weight_counts, WEIGHT_COUNT_NAMES
        )

        device = linear.weight.device
        self.register_buffer(
            "_activation_counts",
            _empty_activation_counts(device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_candidate_histogram",
            torch.zeros(
                config.block_size + 1, dtype=torch.int64, device=device
            ),
            persistent=False,
        )
        self.register_buffer(
            "_activation_encoded_histogram",
            torch.zeros(
                config.block_size + 1, dtype=torch.int64, device=device
            ),
            persistent=False,
        )
        self.register_buffer(
            "_activation_calls",
            torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_sum",
            torch.zeros((), dtype=torch.float64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_min",
            torch.tensor(float("inf"), dtype=torch.float64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_max",
            torch.tensor(float("-inf"), dtype=torch.float64, device=device),
            persistent=False,
        )

    def forward(self, x):
        (
            x_bie,
            threshold,
            counts,
            candidate_histogram,
            encoded_histogram,
        ) = quantize_activation_bie(
            x, self.config, self.config.activation_chunk_rows
        )
        threshold64 = threshold.to(torch.float64)
        self._activation_counts.add_(counts)
        self._activation_candidate_histogram.add_(candidate_histogram)
        self._activation_encoded_histogram.add_(encoded_histogram)
        self._activation_calls.add_(1)
        self._activation_threshold_sum.add_(threshold64)
        self._activation_threshold_min.copy_(
            torch.minimum(self._activation_threshold_min, threshold64)
        )
        self._activation_threshold_max.copy_(
            torch.maximum(self._activation_threshold_max, threshold64)
        )
        return F.linear(
            x_bie, self.linear.weight, self.linear.bias
        ).to(torch.float16)

    def export_stats(self):
        calls = int(self._activation_calls.detach().cpu().item())
        threshold_summary = {
            "count": calls,
            "mean": (
                None
                if calls == 0
                else float(
                    self._activation_threshold_sum.detach().cpu().item() / calls
                )
            ),
            "min": (
                None
                if calls == 0
                else float(self._activation_threshold_min.detach().cpu().item())
            ),
            "max": (
                None
                if calls == 0
                else float(self._activation_threshold_max.detach().cpu().item())
            ),
        }
        activation_counts = _counts_to_dict(
            self._activation_counts, ACTIVATION_COUNT_NAMES
        )
        candidate_histogram = (
            self._activation_candidate_histogram.detach().cpu().tolist()
        )
        encoded_histogram = (
            self._activation_encoded_histogram.detach().cpu().tolist()
        )
        return {
            "weight": {"counts": self.weight_counts},
            "activation": {
                "threshold_summary": threshold_summary,
                **_summarize_activation(
                    activation_counts,
                    candidate_histogram,
                    encoded_histogram,
                    include_tail=False,
                ),
            },
        }


def replace_linear_layers(module, config, prefix=""):
    replaced = {}

    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name

        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            if child.in_features % config.block_size != 0:
                raise ValueError(
                    f"{full_name}: in_features must be divisible by G{config.block_size}."
                )
            weight_counts = quantize_weight_in_place(child.weight, config)
            wrapper = HybridLinear(child, config, weight_counts)
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(
                replace_linear_layers(child, config, full_name)
            )

    return replaced


def export_quantization_stats(replaced):
    weight_totals = {name: 0 for name in WEIGHT_COUNT_NAMES}
    activation_totals = {name: 0 for name in ACTIVATION_COUNT_NAMES}
    aggregate_candidate_histogram = [0] * (CONFIG.block_size + 1)
    aggregate_encoded_histogram = [0] * (CONFIG.block_size + 1)
    activation_threshold_sum = 0.0
    activation_threshold_count = 0
    activation_threshold_min = float("inf")
    activation_threshold_max = float("-inf")
    layers = []

    for name in sorted(replaced):
        layer_stats = replaced[name].export_stats()
        layers.append({"layer_name": name, **layer_stats})

        for key in WEIGHT_COUNT_NAMES:
            weight_totals[key] += layer_stats["weight"]["counts"][key]
        for key in ACTIVATION_COUNT_NAMES:
            activation_totals[key] += layer_stats["activation"]["counts"][key]
        for index, value in enumerate(
            layer_stats["activation"]["candidate_outliers_per_block_histogram"]
        ):
            aggregate_candidate_histogram[index] += value
        for index, value in enumerate(
            layer_stats["activation"]["encoded_outliers_per_block_histogram"]
        ):
            aggregate_encoded_histogram[index] += value

        threshold = layer_stats["activation"]["threshold_summary"]
        if threshold["count"]:
            activation_threshold_sum += threshold["mean"] * threshold["count"]
            activation_threshold_count += threshold["count"]
            activation_threshold_min = min(
                activation_threshold_min, threshold["min"]
            )
            activation_threshold_max = max(
                activation_threshold_max, threshold["max"]
            )

    threshold_summary = {
        "count": activation_threshold_count,
        "mean": (
            None
            if activation_threshold_count == 0
            else activation_threshold_sum / activation_threshold_count
        ),
        "min": (
            None if activation_threshold_count == 0 else activation_threshold_min
        ),
        "max": (
            None if activation_threshold_count == 0 else activation_threshold_max
        ),
    }

    return {
        "aggregate": {
            "weight": {
                "counts": weight_totals,
                "shared_exponents_per_block": 1,
            },
            "activation": {
                "threshold_summary": threshold_summary,
                **_summarize_activation(
                    activation_totals,
                    aggregate_candidate_histogram,
                    aggregate_encoded_histogram,
                    include_tail=True,
                ),
            },
        },
        "layers": layers,
    }

## Synthetic validation

The checks below cover signed-threshold semantics, single-exponent weight quantization, deterministic Top-2 selection, 0/1/2/3/16 candidates, demotion to normal, padding, chunk invariance, exact counter closure, and the Linear wrapper.

In [ ]:
_test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_weight = torch.tensor(
    [[0.0, -1.0, 0.5, 1.5] + [0.0] * 12],
    dtype=torch.float32,
    device=_test_device,
)
_weight_q = _quantize_weight_bfp_rows(_weight, CONFIG)
_weight_max = _weight.abs().amax()
_weight_exp = torch.floor(torch.log2(_weight_max))
_weight_step = torch.pow(
    torch.tensor(2.0, device=_test_device),
    _weight_exp - (CONFIG.mantissa_bits - 1),
)
_weight_reference = (
    torch.round(_weight / _weight_step)
    .clamp(-(1 << CONFIG.mantissa_bits) + 1, (1 << CONFIG.mantissa_bits) - 1)
    * _weight_step
)
assert torch.equal(_weight_q, _weight_reference)

_signed = torch.zeros((1, 16), dtype=torch.float32, device=_test_device)
_signed[0, -1] = -100.0
(
    _signed_q,
    _signed_t,
    _signed_counts,
    _signed_candidate_histogram,
    _signed_encoded_histogram,
) = (
    quantize_activation_bie(_signed, CONFIG, chunk_rows=1)
)
_signed_expected_t = (
    _signed.mean() + CONFIG.sigma_k * _signed.std(unbiased=False)
)
_abs_statistics_t = (
    _signed.abs().mean()
    + CONFIG.sigma_k * _signed.abs().std(unbiased=False)
)
assert torch.allclose(_signed_t, _signed_expected_t, rtol=1e-6, atol=1e-6)
assert not torch.allclose(_signed_t, _abs_statistics_t, rtol=1e-6, atol=1e-6)
assert _signed_counts[1].item() == 1
assert _signed_counts[2].item() == 1
assert _signed_candidate_histogram[1].item() == 1
assert _signed_encoded_histogram[1].item() == 1
assert _signed_q[0, -1] != 0

_crafted = torch.zeros((4, 16), dtype=torch.float32, device=_test_device)
_crafted[1, 0] = 1.0
_crafted[2, :2] = 1.0
_crafted[3, :] = torch.arange(
    1, 17, dtype=torch.float32, device=_test_device
)
(
    _crafted_q,
    _crafted_counts,
    _crafted_candidate_histogram,
    _crafted_encoded_histogram,
) = (
    _quantize_activation_bie_rows_with_threshold(
        _crafted,
        torch.tensor(0.5, device=_test_device),
        CONFIG,
    )
)
_crafted_counts_dict = _counts_to_dict(
    _crafted_counts, ACTIVATION_COUNT_NAMES
)
_crafted_summary = _summarize_activation(
    _crafted_counts_dict,
    _crafted_candidate_histogram.detach().cpu().tolist(),
    _crafted_encoded_histogram.detach().cpu().tolist(),
    include_tail=True,
)
assert _crafted_summary["candidate_outliers_per_block_histogram"][0] == 1
assert _crafted_summary["candidate_outliers_per_block_histogram"][1] == 1
assert _crafted_summary["candidate_outliers_per_block_histogram"][2] == 1
assert _crafted_summary["candidate_outliers_per_block_histogram"][16] == 1
assert _crafted_summary["encoded_outliers_per_block_histogram"][0] == 1
assert _crafted_summary["encoded_outliers_per_block_histogram"][1] == 1
assert _crafted_summary["encoded_outliers_per_block_histogram"][2] == 2
assert _crafted_summary["max_candidate_outliers_per_block"] == 16
assert _crafted_summary["max_encoded_outliers_per_block"] == 2
assert _crafted_counts_dict["candidate_values"] == 19
assert _crafted_counts_dict["encoded_outlier_values"] == 5
assert _crafted_counts_dict["demoted_values"] == 14
assert _crafted_counts_dict["candidate_affected_blocks"] == 3
assert _crafted_counts_dict["cap_triggered_blocks"] == 1
assert _crafted_q[3, 13].item() == 14.0

_tie_magnitude = torch.tensor(
    [[[5.0, 5.0, 5.0] + [0.0] * 13]], device=_test_device
)
_tie_candidate = _tie_magnitude > 0
_tie_selected = _select_top2_candidates(
    _tie_magnitude, _tie_candidate, CONFIG
)
assert _tie_selected[0, 0, :3].detach().cpu().tolist() == [True, True, False]

_partial = torch.tensor(
    [[1.0, 2.0, 3.0]], dtype=torch.float32, device=_test_device
)
_, _partial_counts, _partial_candidate_hist, _partial_encoded_hist = (
    _quantize_activation_bie_rows_with_threshold(
        _partial, torch.tensor(0.5, device=_test_device), CONFIG
    )
)
_partial_counts_dict = _counts_to_dict(
    _partial_counts, ACTIVATION_COUNT_NAMES
)
assert _partial_counts_dict["candidate_values"] == 3
assert _partial_counts_dict["encoded_outlier_values"] == 2
assert _partial_counts_dict["demoted_values"] == 1
assert _partial_candidate_hist[3].item() == 1
assert _partial_encoded_hist[2].item() == 1

torch.manual_seed(7)
_chunk_input = torch.randn(
    (5, 32), dtype=torch.float32, device=_test_device
)
_chunk_q1, _chunk_t1, _chunk_c1, _chunk_ch1, _chunk_eh1 = quantize_activation_bie(
    _chunk_input, CONFIG, chunk_rows=1
)
_chunk_q5, _chunk_t5, _chunk_c5, _chunk_ch5, _chunk_eh5 = quantize_activation_bie(
    _chunk_input, CONFIG, chunk_rows=5
)
assert torch.allclose(_chunk_t1, _chunk_t5, rtol=1e-6, atol=1e-6)
assert torch.equal(_chunk_q1, _chunk_q5)
assert torch.equal(_chunk_c1, _chunk_c5)
assert torch.equal(_chunk_ch1, _chunk_ch5)
assert torch.equal(_chunk_eh1, _chunk_eh5)

_toy_dtype = torch.float16 if _test_device.type == "cuda" else torch.float32
_toy = nn.Sequential(
    nn.Linear(16, 8, bias=False, device=_test_device, dtype=_toy_dtype)
)
_toy_layers = replace_linear_layers(_toy, CONFIG)
_toy_x = torch.randn((2, 16), device=_test_device, dtype=_toy_dtype)
_toy_y = _toy(_toy_x)
_toy_stats = next(iter(_toy_layers.values())).export_stats()
assert len(_toy_layers) == 1
assert _toy_y.shape == (2, 8)
assert torch.isfinite(_toy_y).all()
assert _toy_stats["weight"]["counts"]["total_values"] == 8 * 16
assert _toy_stats["weight"]["counts"]["total_blocks"] == 8
assert _toy_stats["activation"]["counts"]["total_values"] == 2 * 16
assert _toy_stats["activation"]["counts"]["total_blocks"] == 2
assert _toy_stats["activation"]["max_encoded_outliers_per_block"] <= 2
assert _toy_stats["activation"]["threshold_summary"]["count"] == 1

del (
    _weight,
    _weight_q,
    _weight_reference,
    _signed,
    _signed_q,
    _crafted,
    _crafted_q,
    _tie_magnitude,
    _tie_candidate,
    _tie_selected,
    _partial,
    _chunk_input,
    _toy,
    _toy_x,
    _toy_y,
)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Synthetic W-BFP4 / Top-2-capped A-BiE4 checks passed.")

## Dataset

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass("HF_TOKEN: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

## Non-overlapping WikiText-2 perplexity

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("This evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(
        range(0, usable_length, stride),
        total=total_blocks,
        desc="Evaluating W-BFP4 / A-BiE4 G16",
    ):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()
        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": (
            torch.cuda.max_memory_allocated(device) / 2**30
        ),
    }

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("The full PPL evaluation requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

hybrid_layers = replace_linear_layers(model, CONFIG)
if "lm_head" in hybrid_layers or not isinstance(model.lm_head, nn.Linear):
    raise RuntimeError("lm_head must remain an unwrapped FP16 nn.Linear.")
if len(hybrid_layers) != EXPECTED_LINEAR_LAYERS:
    raise RuntimeError(
        f"Expected {EXPECTED_LINEAR_LAYERS} quantized decoder Linear layers, "
        f"got {len(hybrid_layers)}."
    )

parameter_dtypes = {
    parameter.dtype
    for parameter in model.parameters()
    if parameter.is_floating_point()
}
parameter_devices = {
    parameter.device.type for parameter in model.parameters()
}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {"cuda"}, parameter_devices
torch.cuda.empty_cache()

print(f"Hybrid-quantized Linear layers: {len(hybrid_layers)}")
print("Weight: BFP4 G16, one E5 shared exponent")
print("Activation: BiE4 G16, two E5 shared exponents + type bit")
print("lm_head: FP16")

In [ ]:
metrics = evaluate_perplexity(
    model,
    input_ids,
    CONTEXT_LENGTH,
    STRIDE,
    DROP_REMAINDER,
)
quantization_stats = export_quantization_stats(hybrid_layers)
weight_counts = quantization_stats["aggregate"]["weight"]["counts"]
activation_stats = quantization_stats["aggregate"]["activation"]
activation_counts = activation_stats["counts"]

if metrics["evaluated_blocks"] != EXPECTED_EVALUATED_BLOCKS:
    raise RuntimeError("Unexpected number of evaluated WikiText-2 blocks.")
if metrics["evaluated_tokens"] != EXPECTED_LOSS_TOKENS:
    raise RuntimeError("Unexpected number of evaluated loss tokens.")
if weight_counts["total_values"] != EXPECTED_WEIGHT_VALUES:
    raise RuntimeError("Unexpected number of quantized weight values.")
if activation_counts["total_values"] != EXPECTED_ACTIVATION_VALUES:
    raise RuntimeError("Unexpected number of quantized activation values.")
if weight_counts["total_blocks"] * CONFIG.block_size != weight_counts["total_values"]:
    raise RuntimeError("Weight block/value closure failed.")
if activation_counts["total_blocks"] * CONFIG.block_size != activation_counts["total_values"]:
    raise RuntimeError("Activation block/value closure failed.")
if activation_stats["threshold_summary"]["count"] != (
    EXPECTED_LINEAR_LAYERS * EXPECTED_EVALUATED_BLOCKS
):
    raise RuntimeError("Unexpected number of activation threshold evaluations.")
if activation_counts["candidate_values"] != (
    activation_counts["encoded_outlier_values"]
    + activation_counts["demoted_values"]
):
    raise RuntimeError("Candidate/encoded/demoted aggregate closure failed.")
if activation_stats["max_encoded_outliers_per_block"] > (
    CONFIG.max_outliers_per_block
):
    raise RuntimeError("Encoded activation occupancy exceeds Top-2.")

result = {
    "model": MODEL_ID,
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "split": SPLIT,
    "quantization": "W-BFP4 / Top-2-capped A-BiE4 fake quantization",
    "method_label": "activation-only BiE signed-3sigma Top-2 cap ablation",
    "format": (
        f"Weight BFP4 (1S{CONFIG.mantissa_bits}M + one E{CONFIG.shared_exponent_bits}, "
        f"G{CONFIG.block_size}); Activation BiE4 (1S{CONFIG.mantissa_bits}M + two "
        f"E{CONFIG.shared_exponent_bits} + capped type, G{CONFIG.block_size})"
    ),
    "hybrid_config": asdict(CONFIG),
    "threshold_contract": {
        "domain": "activation only",
        "formula": "mean(X) + sigma_k * std(X)",
        "statistics_domain": "signed tensor values",
        "comparison": "candidate iff abs(X) > threshold",
        "std_correction": 0,
        "granularity": "one complete nn.Linear input tensor per forward call",
        "computed_before_row_chunking": True,
        "weight_threshold_or_outlier_classification": False,
    },
    "activation_block_outlier_contract": {
        "block_definition": "contiguous Group-16 values along Linear K dimension",
        "classification_stage": "candidate detection before Top-2 selection",
        "candidate_rule": "abs(X) > threshold",
        "encoded_rule": "up to two largest-magnitude candidates per block",
        "demoted_rule": "remaining candidates join normal before exponent selection",
        "tie_break": CONFIG.topk_tie_break,
        "maximum_candidate_outliers_per_full_block": CONFIG.block_size,
        "maximum_encoded_outliers_per_full_block": (
            CONFIG.max_outliers_per_block
        ),
    },
    "storage_contract": {
        "packed_storage_implemented": False,
        "sparse_index_storage_implemented": False,
        "tensor_threshold_metadata_excluded": True,
        "weight": {
            "private_value_bits": PRIVATE_VALUE_BITS,
            "type_bits_per_value": 0,
            "shared_exponents_per_block": 1,
            "shared_exponent_bits": CONFIG.shared_exponent_bits,
            "effective_bits_per_value": WEIGHT_BITS_PER_VALUE,
        },
        "activation": {
            "private_value_bits": PRIVATE_VALUE_BITS,
            "type_bits_per_value": 1,
            "shared_exponents_per_block": 2,
            "shared_exponent_bits": CONFIG.shared_exponent_bits,
            "effective_bits_per_value": ACTIVATION_BITS_PER_VALUE,
            "maximum_encoded_outliers_per_block": (
                CONFIG.max_outliers_per_block
            ),
        },
    },
    "quantized_scope": {
        "operations": "LLaMA decoder nn.Linear modules",
        "weights": "BFP4 single shared exponent",
        "activations": "Top-2-capped BiE4 dual shared exponent",
        "lm_head": False,
        "attention_internal_matmul": False,
        "softmax": False,
        "layernorm": False,
        "dewa": False,
        "fp_acc_exception_routing": False,
    },
    "linear_output_dtype": "float16",
    "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
    "quantized_linear_layers": len(hybrid_layers),
    "attention_implementation": "eager",
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "drop_remainder": DROP_REMAINDER,
    "baseline_perplexity": FP16_BASELINE_PPL,
    "delta_perplexity": metrics["perplexity"] - FP16_BASELINE_PPL,
    "ppl_references": {
        "fp16": FP16_BASELINE_PPL,
        "w_bie4_a_bie4_g16_signed_mu3sigma": WA_BIE4_BASELINE_PPL,
        "w_bfp4_a_bie4_g16_signed_mu3sigma_uncapped": (
            ACTIVATION_BIE4_BASELINE_PPL
        ),
    },
    "delta_perplexity_to_fp16": metrics["perplexity"] - FP16_BASELINE_PPL,
    "delta_perplexity_to_w_bie4_a_bie4": (
        metrics["perplexity"] - WA_BIE4_BASELINE_PPL
    ),
    "delta_perplexity_to_uncapped_activation_bie4": (
        metrics["perplexity"] - ACTIVATION_BIE4_BASELINE_PPL
    ),
    "quantization_stats": quantization_stats,
    "validation": {
        "expected_quantized_linear_layers": EXPECTED_LINEAR_LAYERS,
        "expected_evaluated_blocks": EXPECTED_EVALUATED_BLOCKS,
        "expected_loss_tokens": EXPECTED_LOSS_TOKENS,
        "expected_weight_values": EXPECTED_WEIGHT_VALUES,
        "expected_activation_values": EXPECTED_ACTIVATION_VALUES,
        "all_checks_passed": True,
    },
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    **metrics,
}

OUTPUT_PATH.write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(json.dumps({
    "perplexity": result["perplexity"],
    "delta_perplexity_to_fp16": result["delta_perplexity_to_fp16"],
    "delta_perplexity_to_w_bie4_a_bie4": result[
        "delta_perplexity_to_w_bie4_a_bie4"
    ],
    "delta_perplexity_to_uncapped_activation_bie4": result[
        "delta_perplexity_to_uncapped_activation_bie4"
    ],
    "candidate_value_rate": activation_stats["rates"][
        "candidate_value_rate"
    ],
    "encoded_outlier_value_rate": activation_stats["rates"][
        "encoded_outlier_value_rate"
    ],
    "demoted_value_rate": activation_stats["rates"][
        "demoted_value_rate"
    ],
    "cap_triggered_block_rate": activation_stats["rates"][
        "cap_triggered_block_rate"
    ],
    "max_candidate_outliers_per_activation_block": activation_stats[
        "max_candidate_outliers_per_block"
    ],
    "max_encoded_outliers_per_activation_block": activation_stats[
        "max_encoded_outliers_per_block"
    ],
}, indent=2, ensure_ascii=False))
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"Result JSON does not exist: {OUTPUT_PATH}")

try:
    from google.colab import files
except ImportError:
    print(f"Not running in Colab. JSON remains at: {OUTPUT_PATH.resolve()}")
else:
    files.download(str(OUTPUT_PATH))